In [2]:
import ee

# NOTA: "viirs-peru" es el ID del proyecto de Google Cloud vinculado a Earth Engine, el nombre fue ese pero es para todo EARTH ENGINE
# sirve como autenticación para CUALQUIER dataset de Earth Engine (WorldCover, SRTM,
# Sentinel-2, Open Buildings, etc.), no solo para VIIRS.

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")

Google Earth Engine listo


In [3]:
from pathlib import Path
import geopandas as gpd
import json
import pandas as pd
import time

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"

geobase = gpd.read_file(DATA / "clean" / "staging" / "geobase_distrital.gpkg")
print("Distritos en geobase:", len(geobase))

Distritos en geobase: 1889


## Google Open Buildings V3 — Conteo y densidad de edificaciones

**Fuente:** Google Research. Disponible en Google Earth Engine como
`GOOGLE/Research/open-buildings/v3/polygons`. Documentación:
https://sites.research.google/gr/open-buildings/

**Qué es, en simple:** una capa con el contorno (el polígono) de cada edificación
detectada en imágenes satelitales, para más de mil millones de construcciones en el
mundo. No es un mapa de "aquí hay zona construida sí/no" como World Cover — es un
inventario, edificación por edificación, con su forma, su área y qué tan seguro está el
modelo de que ahí realmente hay una construcción. La versión 3 (a diferencia de la v2, que
solo cubría África) ya incluye Latinoamérica y el Caribe, así que Perú está cubierto.

**Años cubiertos:** no es un panel por año. Es una sola capa "actual", construida a partir
de imágenes cuya fecha de captura varía según la región (no hay una fecha única global de
referencia). Al igual que con SRTM y World Cover, para el panel del proyecto (2021-2025)
se toma este único valor disponible y se repite igual para los cinco años — no existe una
versión 2021, otra 2022, etc.

**Por qué importa para anemia:** es la variable de contexto más directamente ligada a la
hipótesis central del proyecto: la dispersión de viviendas. Un distrito con casas muy
separadas entre sí hace mucho más difícil el seguimiento domiciliario y la adherencia a la
suplementación de hierro que uno con viviendas concentradas — es exactamente el tipo de
factor que puede explicar por qué el mismo gasto rinde distinto según el territorio. A
nivel distrital, esta capa da una primera aproximación (conteo y densidad de
edificaciones); la dispersión fina a nivel de caserío individual la dará después la
segmentación semántica de la Etapa 2, sobre imágenes Planet NICFI.

**Variables extraídas (por distrito):**

| Variable | Qué mide |
|---|---|
| `n_edificios` | Cuántas edificaciones detectó el modelo dentro del distrito |
| `area_construida_m2` | Suma del área de todas esas edificaciones |
| `densidad_edificios_km2` | `n_edificios` entre el área del distrito — entre más bajo, más disperso el poblamiento |
| `confianza_media` | Promedio del puntaje de confianza de las detecciones (más bajo en construcciones pequeñas o techos de material no estándar, típico de zona rural) |

**Método:** a diferencia de SRTM, JRC y World Cover (que son imágenes y se agregan con
`reduceRegions` promediando píxeles), Open Buildings V3 es una colección de polígonos
(vectores). Por distrito, se filtran qué edificaciones caen dentro de su geometría
(`filterBounds`) y se cuentan/suman sus atributos (`size()`, `aggregate_sum`,
`aggregate_mean`). Se procesa igual en lotes de 40 distritos para no saturar Earth Engine,
aunque esta operación es más pesada que las anteriores porque cada distrito se compara
contra una capa de más de mil millones de polígonos a nivel global — si algún lote falla
por timeout, conviene bajar el tamaño de lote a 20 o 10.

**¿Usa IA? Sí — pero no la corren ustedes.** Google entrenó un modelo de deep learning
(detección de objetos sobre imágenes satelitales) para generar estos polígonos, y publicó
el resultado ya calculado. El equipo del proyecto no entrena ni corre ese modelo: solo
consulta la capa de edificaciones ya detectadas, igual que con World Cover. La única IA que
el proyecto sí entrena/ajusta directamente es la segmentación de la Etapa 2 (U-Net /
SegFormer, con apoyo de SAM para etiquetar).

**Estado actual: no iniciado.** En el notebook solo hay una llamada de prueba
(`open_buildings.limit(5).getInfo()`) para confirmar que la fuente carga — falta escribir
la extracción real por distrito (el bloque de código de arriba) y correrla para el rango
completo.

In [4]:
# ============================================================================
# Google Open Buildings V3 — conteo, área construida y densidad por distrito
# ============================================================================
#
# Qué es esta fuente:
# Google entrenó un modelo de detección de objetos (deep learning) sobre
# imágenes satelitales para encontrar el contorno de cada edificación en el
# planeta. El resultado ya viene calculado y publicado como una capa de
# polígonos — acá NO se entrena ni se corre ningún modelo, solo se consulta
# la salida ya generada por Google, igual que con ESA World Cover.
#
# Diferencia clave con SRTM/JRC/World Cover:
# Esas tres fuentes son IMÁGENES (un valor por píxel), así que se agregan
# con reduceRegions promediando píxeles dentro del polígono distrital.
# Open Buildings V3 es una colección de POLÍGONOS (vectores) — acá no hay
# píxeles que promediar, hay que filtrar qué edificaciones caen dentro de
# cada distrito y contar/sumar sus atributos. Por eso el patrón de código
# es distinto (filterBounds + size/aggregate en vez de reduceRegions).
#
# Cobertura: v3 incluye Latinoamérica y el Caribe (a diferencia de v2, que
# era solo África), así que Perú está cubierto.
#
# Temporalidad: NO es un panel por año. Es una sola capa "actual" (la fecha
# de captura de las imágenes de origen varía por región, no hay una fecha
# única global). Igual que con SRTM y World Cover, este valor se repite para
# los 5 años del panel del proyecto (2021-2025) porque la fuente no tiene
# esa granularidad temporal.
# ============================================================================

import ee

# -----------------------------------------------------------------------
# 1. Cargar la colección de edificaciones ya detectadas
# -----------------------------------------------------------------------
# Cada feature trae, entre otras propiedades:
#   - area_in_meters: área del polígono de la edificación
#   - confidence: qué tan seguro está el modelo de que ahí hay una
#     edificación real (0 a 1). Más bajo en construcciones pequeñas o con
#     techos de material poco estándar — típico de zona rural.
open_buildings = ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")


# -----------------------------------------------------------------------
# 2. Función de conteo/agregación por distrito
# -----------------------------------------------------------------------
# Se aplica con .map() sobre cada feature (distrito) de un lote. Todo lo
# que pasa adentro corre del lado del servidor de Earth Engine — no se
# hace ningún .getInfo() aquí dentro, eso es clave para que sea eficiente.
def contar_edificios(feature):
    geom = feature.geometry()

    # Filtrar del total global solo las edificaciones que caen dentro
    # de este distrito. Esta es la operación más pesada: compara contra
    # más de mil millones de polígonos a nivel mundial.
    edificios_en_distrito = open_buildings.filterBounds(geom)

    # Área del distrito calculada de forma geodésica (en metros cuadrados,
    # sobre la superficie real de la Tierra), NO con geopandas.
    # ¿Por qué no usar geometry.area de geopandas? Porque si el shapefile
    # está en EPSG:4326 (lat/lon, el CRS por defecto de la mayoría de
    # shapefiles públicos), .area devuelve un número en grados cuadrados,
    # que no es un área real y da densidades sin sentido. Calculándolo acá
    # con Earth Engine se evita ese error por completo.
    area_distrito_m2 = geom.area(maxError=1)

    return feature.set({
        "n_edificios": edificios_en_distrito.size(),
        "area_construida_m2": edificios_en_distrito.aggregate_sum("area_in_meters"),
        "confianza_media": edificios_en_distrito.aggregate_mean("confidence"),
        "area_distrito_km2": ee.Number(area_distrito_m2).divide(1e6),
    })

In [5]:
# -----------------------------------------------------------------------
# 3. Procesamiento por lotes de 40 distritos
# -----------------------------------------------------------------------
# Mismo patrón que ya usaron en SRTM/JRC/NDVI: se procesa en lotes en vez
# de mandar los ~1,889 distritos de una sola vez, porque Earth Engine
# limita el tamaño de la respuesta (~10MB) por request.
#
# A diferencia de JRC, esta operación es más pesada porque cada distrito
# se compara contra una capa global de más de mil millones de polígonos.
# Por eso se agrega manejo de errores por lote: si un lote falla, no se
# pierde todo el progreso — se reporta y se sigue con el siguiente.
#
# Nota: acá NO se simplifica la geometría (a diferencia de SRTM/JRC/NDVI)
# porque el conteo de edificios depende de un filtro espacial exacto
# (filterBounds) — simplificar el borde del distrito podría perder o
# duplicar edificaciones cerca del límite con el distrito vecino.
#
# Si varios lotes fallan por timeout, bajar tamano_lote a 20 o 10 suele
# resolverlo (menos distritos por request = menos probabilidad de timeout).
distritos_base = geobase[["UBIGEO", "geometry"]].copy()
distritos_base = distritos_base[
    distritos_base.geometry.notna() & ~distritos_base.geometry.is_empty
].copy()

lista_ubigeos = distritos_base["UBIGEO"].tolist()
n_distritos = len(lista_ubigeos)
tamano_lote = 40
n_lotes = -(-n_distritos // tamano_lote)  # división hacia arriba

resultados_ob = []
lotes_fallidos = []

for i in range(0, n_distritos, tamano_lote):
    lote_ids = lista_ubigeos[i:i + tamano_lote]

    # Filtrar el lote en geopandas y convertir SOLO ese lote a
    # ee.FeatureCollection (no todo el shapefile de una vez).
    gdf_lote = distritos_base[distritos_base["UBIGEO"].isin(lote_ids)]
    geojson_lote = json.loads(gdf_lote.to_json())
    fc_lote = ee.FeatureCollection(geojson_lote)

    try:
        fc_resultado = fc_lote.map(contar_edificios)

        datos_lote = fc_resultado.reduceColumns(
            ee.Reducer.toList(5),
            ["UBIGEO", "n_edificios", "area_construida_m2",
             "confianza_media", "area_distrito_km2"]
        ).get("list").getInfo()

        resultados_ob.extend(datos_lote)
        print(f"Lote {i//tamano_lote + 1}/{n_lotes} — "
              f"distritos {i} a {min(i+tamano_lote, n_distritos)} listo ✓")

    except Exception as e:
        print(f"  ⚠️  Error en lote {i//tamano_lote + 1}/{n_lotes}: {e}")
        lotes_fallidos.append((i, min(i + tamano_lote, n_distritos)))
        continue

    # Pequeño respiro para no saturar la API de GEE, igual que en NDVI.
    time.sleep(0.2)

print("\nExtracción completa.")
if lotes_fallidos:
    print(f"⚠️  {len(lotes_fallidos)} lote(s) fallaron y quedaron pendientes de reintentar: {lotes_fallidos}")

Lote 1/48 — distritos 0 a 40 listo ✓
Lote 2/48 — distritos 40 a 80 listo ✓
Lote 3/48 — distritos 80 a 120 listo ✓
Lote 4/48 — distritos 120 a 160 listo ✓
Lote 5/48 — distritos 160 a 200 listo ✓
Lote 6/48 — distritos 200 a 240 listo ✓
Lote 7/48 — distritos 240 a 280 listo ✓
Lote 8/48 — distritos 280 a 320 listo ✓
Lote 9/48 — distritos 320 a 360 listo ✓
Lote 10/48 — distritos 360 a 400 listo ✓
Lote 11/48 — distritos 400 a 440 listo ✓
Lote 12/48 — distritos 440 a 480 listo ✓
Lote 13/48 — distritos 480 a 520 listo ✓
Lote 14/48 — distritos 520 a 560 listo ✓
Lote 15/48 — distritos 560 a 600 listo ✓
Lote 16/48 — distritos 600 a 640 listo ✓
Lote 17/48 — distritos 640 a 680 listo ✓
Lote 18/48 — distritos 680 a 720 listo ✓
Lote 19/48 — distritos 720 a 760 listo ✓
Lote 20/48 — distritos 760 a 800 listo ✓
Lote 21/48 — distritos 800 a 840 listo ✓
Lote 22/48 — distritos 840 a 880 listo ✓
Lote 23/48 — distritos 880 a 920 listo ✓
Lote 24/48 — distritos 920 a 960 listo ✓
Lote 25/48 — distritos 960 a 10

In [6]:
df_open_buildings = pd.DataFrame(
    resultados_ob,
    columns=["ubigeo", "n_edificios", "area_construida_m2",
             "confianza_media", "area_distrito_km2"]
)

# Densidad de edificaciones por km² — proxy de dispersión de viviendas a
# nivel distrital: entre más baja, más disperso el poblamiento (más difícil
# el seguimiento domiciliario para los programas de suplementación de
# hierro). La dispersión fina a nivel de caserío la dará después la
# segmentación de la Etapa 2.
df_open_buildings["densidad_edificios_km2"] = (
    df_open_buildings["n_edificios"] / df_open_buildings["area_distrito_km2"]
)

# Validación rápida: distritos con 0 edificios detectados son una señal a
# revisar (puede ser un distrito genuinamente sin población, o un fallo de
# detección del modelo en zonas de nubosidad persistente o vegetación densa).
sin_edificios = df_open_buildings[df_open_buildings["n_edificios"] == 0]
if len(sin_edificios) > 0:
    print(f"\n⚠️  {len(sin_edificios)} distrito(s) con 0 edificios detectados — revisar:")
    print(sin_edificios[["ubigeo", "n_edificios"]])

print(f"\n{df_open_buildings.shape}")
df_open_buildings.describe()


(1889, 6)


,n_edificios,area_construida_m2,confianza_media,area_distrito_km2,densidad_edificios_km2
count,1889.000000,1.889000e+03,1889.000000,1889.000000,1889.000000
mean,10830.542615,7.851509e+05,0.780874,683.576375,146.489385
std,21665.500401,1.723068e+06,0.012508,1894.484179,526.479915
min,65.000000,2.341142e+03,0.741678,0.796330,0.036080
25%,1850.000000,1.152335e+05,0.772520,94.352423,7.191601
50%,4358.000000,2.655245e+05,0.780387,205.704726,19.618664
75%,10202.000000,6.697196e+05,0.788581,501.878930,51.232922
max,370453.000000,2.798795e+07,0.833320,24279.509117,6613.444949


## guardar

In [7]:
output_path = DATA / "clean" / "staging" / "open_buildings_distrital.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_open_buildings.to_csv(output_path, index=False)
print("Guardado en:", output_path)

Guardado en: c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model\data\clean\staging\open_buildings_distrital.csv
